# Train QDD on Novozymes Enzyme Stability

**Run this notebook on Google Colab Pro** (A100/V100 recommended).

This notebook:
1. Clones the repo and installs dependencies
2. Downloads Novozymes competition data from Kaggle
3. Precomputes VDOS for the wildtype structure
4. Trains `VibroStructuralModel` with early stopping
5. Evaluates and saves the checkpoint
6. Generates a Kaggle submission CSV

**Time estimate: ~15-30 minutes on A100**

## 1. Setup

In [ ]:
# Clone repository
!git clone https://github.com/jayhemnani9910/nobel-dataintelligence.git
%cd nobel-dataintelligence

# Install dependencies
!pip install -q -r requirements.txt

# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Download Novozymes Data

You need Kaggle API credentials. Either:
- Upload `kaggle.json` via the file browser (left panel)
- Or use Colab Secrets to set `KAGGLE_USERNAME` and `KAGGLE_KEY`

In [ ]:
import os
from pathlib import Path

# Option 1: Upload kaggle.json
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)

if not (kaggle_dir / 'kaggle.json').exists():
    # Try Colab Secrets first
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
        print("Using Kaggle credentials from Colab Secrets")
    except Exception:
        # Upload manually
        from google.colab import files
        print("Upload your kaggle.json file:")
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            (kaggle_dir / 'kaggle.json').write_bytes(uploaded['kaggle.json'])
            os.chmod(str(kaggle_dir / 'kaggle.json'), 0o600)
            print("Saved kaggle.json")
else:
    print("kaggle.json already exists")

In [ ]:
# Download competition data
DATA_DIR = Path('./data/kaggle')
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not (DATA_DIR / 'train.csv').exists():
    !kaggle competitions download -c novozymes-enzyme-stability-prediction -p {DATA_DIR}
    !cd {DATA_DIR} && unzip -o '*.zip' && rm -f *.zip
    print("Data downloaded and extracted")
else:
    print("Data already exists")

# Verify
for f in ['train.csv', 'test.csv', 'wildtype_structure_prediction_af2.pdb']:
    path = DATA_DIR / f
    print(f"  {'✓' if path.exists() else '✗'} {f} ({path.stat().st_size // 1024}KB)" if path.exists() else f"  ✗ {f} MISSING")

## 3. Explore Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

train_df = pd.read_csv(DATA_DIR / 'train.csv')
print(f"Training samples: {len(train_df)}")
print(f"Columns: {list(train_df.columns)}")
print(f"\nTm distribution:")
print(train_df['tm'].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train_df['tm'].dropna(), bins=50, color='#22d3ee', alpha=0.8)
axes[0].set_xlabel('Tm (°C)'); axes[0].set_ylabel('Count'); axes[0].set_title('Melting Temperature Distribution')
axes[1].hist(train_df['pH'], bins=20, color='#f472b6', alpha=0.8)
axes[1].set_xlabel('pH'); axes[1].set_ylabel('Count'); axes[1].set_title('pH Distribution')
plt.tight_layout(); plt.show()

## 4. Precompute VDOS

In [ ]:
SPECTRA_DIR = Path('./data/spectral')
SPECTRA_DIR.mkdir(parents=True, exist_ok=True)

wt_spectrum_path = SPECTRA_DIR / 'wt_spectrum.npy'
wt_pdb = DATA_DIR / 'wildtype_structure_prediction_af2.pdb'

if not wt_spectrum_path.exists() and wt_pdb.exists():
    try:
        from src.nma_analysis import ANMAnalyzer
        print("Running NMA on wildtype structure...")
        anm = ANMAnalyzer(str(wt_pdb), cutoff=15.0)
        freqs, modes = anm.compute_modes(k=100)
        vdos = anm.compute_vdos(k=100, broadening=5.0)
        s_vib = anm.compute_vibrational_entropy(k=100)
        np.save(wt_spectrum_path, vdos)
        print(f"VDOS saved: {vdos.shape}, S_vib = {s_vib:.2f} J/(mol·K)")
        print(f"Frequency range: {freqs.min():.1f} - {freqs.max():.1f} cm⁻¹")
    except ImportError:
        print("ProDy not available — generating synthetic VDOS")
        vdos = np.random.rand(1000) * 0.5
        np.save(wt_spectrum_path, vdos)
elif wt_spectrum_path.exists():
    vdos = np.load(wt_spectrum_path)
    print(f"Loaded existing VDOS: {vdos.shape}")
else:
    print("No wildtype PDB found — generating zeros")
    vdos = np.zeros(1000)
    np.save(wt_spectrum_path, vdos)

# Plot VDOS
plt.figure(figsize=(10, 3))
plt.plot(np.linspace(0, 500, len(vdos)), vdos, color='#22d3ee', linewidth=1.5)
plt.fill_between(np.linspace(0, 500, len(vdos)), vdos, alpha=0.2, color='#22d3ee')
plt.xlabel('Frequency (cm⁻¹)'); plt.ylabel('Intensity')
plt.title('Wildtype VDOS Spectrum'); plt.tight_layout(); plt.show()

## 5. Create Dataset & DataLoaders

In [ ]:
from torch.utils.data import DataLoader, random_split
from src.datasets import NovozymesDataset
from src.utils import batch_collate_function, set_seed

set_seed(42)

dataset = NovozymesDataset(
    csv_file=str(DATA_DIR / 'train.csv'),
    structure_file=str(wt_pdb),
    spectra_dir=str(SPECTRA_DIR),
    include_updates=True,
)
print(f"Dataset size: {len(dataset)}")

# 80/20 split
n_train = int(0.8 * len(dataset))
n_val = len(dataset) - n_train
train_set, val_set = random_split(dataset, [n_train, n_val])
print(f"Train: {len(train_set)}, Val: {len(val_set)}")

BATCH_SIZE = 16
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, collate_fn=batch_collate_function)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, collate_fn=batch_collate_function)

## 6. Initialize Model

In [ ]:
from src.models.multimodal import VibroStructuralModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = VibroStructuralModel(
    latent_dim=128,
    gnn_input_dim=24,
    fusion_type='bilinear',
    dropout=0.2,
    num_go_terms=100,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Device: {device}")

## 7. Train

In [ ]:
from src.training import Trainer, MetricComputer

EPOCHS = 30
LR = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
loss_fn = torch.nn.MSELoss()

CHECKPOINT_DIR = './checkpoints/novozymes'
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
)

best_loss = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    epochs=EPOCHS,
    metric_fn=MetricComputer.spearman_correlation,
    early_stopping_patience=10,
    task='novozymes',
)
print(f"\nBest validation loss: {best_loss:.4f}")

## 8. Evaluate

In [ ]:
# Evaluate on validation set
val_metrics = trainer.validate(val_loader, loss_fn, metric_fn=MetricComputer.spearman_correlation, task='novozymes')
print(f"Validation Loss: {val_metrics['val_loss']:.4f}")
print(f"Spearman Correlation: {val_metrics.get('metric', 'N/A')}")

# Collect predictions for plot
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for batch in val_loader:
        if batch is None: continue
        graph = batch['graph'].to(device)
        spectra = batch['spectra'].to(device)
        gf = batch.get('global_features')
        if gf is not None: gf = gf.to(device)
        out = model(graph, spectra, global_features=gf, task='novozymes').squeeze(-1)
        all_preds.append(out.cpu().numpy())
        all_targets.append(batch['labels'].numpy())

preds = np.concatenate(all_preds)
targets = np.concatenate(all_targets)

# Correlation plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(targets, preds, alpha=0.3, s=10, color='#22d3ee')
axes[0].plot([targets.min(), targets.max()], [targets.min(), targets.max()], 'r--', alpha=0.5)
axes[0].set_xlabel('Actual Tm'); axes[0].set_ylabel('Predicted Tm')
axes[0].set_title(f'Predictions vs Targets (Spearman={val_metrics.get("metric", 0):.3f})')

errors = preds - targets
axes[1].hist(errors, bins=50, color='#f472b6', alpha=0.8)
axes[1].set_xlabel('Error (Predicted - Actual)'); axes[1].set_ylabel('Count')
axes[1].set_title(f'Error Distribution (MAE={np.mean(np.abs(errors)):.2f})')
plt.tight_layout(); plt.show()

## 9. Save Final Checkpoint

In [ ]:
# Save as the canonical checkpoint name
final_path = Path(CHECKPOINT_DIR) / 'novozymes_best.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'best_loss': best_loss,
    'epoch': trainer.best_epoch,
    'config': {
        'latent_dim': 128,
        'gnn_input_dim': 24,
        'fusion_type': 'bilinear',
        'dropout': 0.2,
    },
}, final_path)
print(f"Checkpoint saved: {final_path} ({final_path.stat().st_size / 1024:.0f} KB)")

## 10. Generate Kaggle Submission

In [ ]:
test_csv = DATA_DIR / 'test.csv'
if test_csv.exists():
    test_dataset = NovozymesDataset(
        csv_file=str(test_csv),
        structure_file=str(wt_pdb),
        spectra_dir=str(SPECTRA_DIR),
        include_updates=False,
    )
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=batch_collate_function)

    model.eval()
    test_preds = []
    with torch.no_grad():
        for batch in test_loader:
            if batch is None: continue
            graph = batch['graph'].to(device)
            spectra = batch['spectra'].to(device)
            gf = batch.get('global_features')
            if gf is not None: gf = gf.to(device)
            out = model(graph, spectra, global_features=gf, task='novozymes').squeeze(-1)
            test_preds.append(out.cpu().numpy())

    test_preds = np.concatenate(test_preds)
    test_df = pd.read_csv(test_csv)
    submission = pd.DataFrame({'seq_id': test_df['seq_id'], 'tm': test_preds})
    submission_path = Path('./submissions/novozymes_submission.csv')
    submission_path.parent.mkdir(exist_ok=True)
    submission.to_csv(submission_path, index=False)
    print(f"Submission saved: {submission_path}")
    print(submission.head())
else:
    print("No test.csv found — skipping submission generation")

## 11. Download Checkpoint

Download the trained checkpoint to use locally for inference.

In [ ]:
try:
    from google.colab import files
    files.download(str(final_path))
    print("Download started!")
except ImportError:
    print(f"Not on Colab. Checkpoint at: {final_path.resolve()}")